# ThermoRG-NN Phase A — Theory Validation Pipeline
## TAS Metrics Profiling + d_manifold + Publication Plots

**流程**: ▶️ Environment → ▶️ Smoke Test → ▶️ Load Training Results → ▶️ TAS Profiling → ▶️ Push to GitHub

---

In [ ]:
import os, sys, subprocess
from kaggle_secrets import UserSecretsClient

## ▶️ Step 1: Environment & Code Clone

In [ ]:
%cd /kaggle/working
!rm -rf ThermoRG-NN
user_secrets = UserSecretsClient()
gh_token = user_secrets.get_secret("gh_token")
repo_url = f"https://{gh_token}@github.com/xliu203/ThermoRG-NN.git"
!git clone {repo_url}
%cd /kaggle/working/ThermoRG-NN
!git config --global user.email "xliu203@asu.edu"
!git config --global user.name "Leo Liu"
print("Code cloned and identity configured")

In [ ]:
!pip install -q torch torchvision scipy matplotlib
sys.path.insert(0, "/kaggle/working/ThermoRG-NN/src")
sys.path.insert(0, "/kaggle/working/ThermoRG-NN")
print("Environment ready")

## ▶️ Step 2: Smoke Test (No GPU, No Real Data)

Catches import errors, d_manifold computation, JSON/CSV writing, matplotlib availability. Does NOT burn GPU quota.

In [ ]:
!python3 experiments/phase_a/smoke_test.py

## ▶️ Step 3: Load Training Results

Reads best validation accuracy from existing `experiments/lift_test/results/<arch>/phase_a_metrics.csv` files. If empty, training has not been run yet.

In [ ]:
import glob, json, os, pandas as pd

RESULTS_DIR = "experiments/phase_a/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Check what training results exist
csv_files = sorted(glob.glob("experiments/lift_test/results/*/phase_a_metrics.csv"))
print(f"Found {len(csv_files)} training result CSV(s):")
for f in csv_files:
    arch = f.split("/")[-2]
    df = pd.read_csv(f)
    acc_col = "val_acc" if "val_acc" in df.columns else "test_acc"
    print(f"  {arch}: best_{acc_col}={df[acc_col].max():.2f}%")

if not csv_files:
    print("WARNING: No training results found. Run training first, or use --use-fake flag.")
    print("You can still run phase_a_analysis.py with --use-fake to test the pipeline.")

# Build training_results.json
actual_results = {}
for csv_path in csv_files:
    arch = csv_path.split("/")[-2]
    df = pd.read_csv(csv_path)
    acc_col = "val_acc" if "val_acc" in df.columns else "test_acc"
    actual_results[arch] = {"best_acc": float(df[acc_col].max())}

with open(f"{RESULTS_DIR}/training_results.json", "w") as f:
    json.dump(actual_results, f, indent=2)
print(f"\nSaved {len(actual_results)} architectures to {RESULTS_DIR}/training_results.json")

## ▶️ Step 4: TAS Profiling (GPU)

Computes d_manifold, profiles all 15 architectures with TAS alpha/J_topo metrics, generates 3 publication plots.

In [ ]:
r = subprocess.run(
    [
        "python3",
        "experiments/phase_a/phase_a_analysis.py",
        "--device", "cuda",
        "--n-samples", "5000",
        "--n-per-arch", "500",
        "--actual-results", f"{RESULTS_DIR}/training_results.json",
        "--output-dir", RESULTS_DIR,
    ],
    capture_output=True,
    text=True,
)
print("=== STDOUT (last 4000 chars) ===")
print(r.stdout[-4000:] if r.stdout else "(empty)")
print("\n=== STDERR (last 1000 chars) ===")
print(r.stderr[-1000:] if r.stderr else "(empty)")
print(f"\nReturn code: {r.returncode}")

## ▶️ Step 5: Push Results to GitHub

Always runs — even on failure. Pushes all CSVs, JSONs, and plots under `experiments/phase_a/results/`.

In [ ]:
cmds = [
    f"git add {RESULTS_DIR}/*.csv {RESULTS_DIR}/*.json {RESULTS_DIR}/*.png 2>/dev/null || true",
    'git commit -m "Phase A results: TAS metrics + plots from Kaggle run" || true',
    'git push origin main 2>&1 || git push origin master 2>&1 || true'
]

for cmd in cmds:
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                         cwd="/kaggle/working/ThermoRG-NN")
    if res.returncode != 0 and "nothing to commit" not in res.stdout and "nothing to commit" not in res.stderr:
        print(f"WARN: {res.stderr[-300:]}")
    else:
        print(f"OK: {cmd[:70]}")

print("\nDone. Results should be visible on GitHub.")